## 0 — GPU + Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE='/content/drive/MyDrive/android-auto-ai-agent'
STORES=f'{DRIVE}/stores'
os.makedirs(STORES,exist_ok=True)
print('STORES =',STORES)

## 1 — Clone project + deps

In [ ]:
import os
if not os.path.isdir('/content/android-auto-ai-agent'):
    !git clone https://github.com/appdev1307/android-auto-ai-agent.git /content/android-auto-ai-agent
else:
    !cd /content/android-auto-ai-agent && git pull
%cd /content/android-auto-ai-agent
!apt-get -qq install -y ripgrep >/dev/null && echo ripgrep ok
!pip -q install -r requirements.txt
print('project ready')

## 2 — Config: stores→Drive, agent→Ollama

In [ ]:
import yaml, pathlib, os
LLM_MODEL='qwen2.5-coder:32b'
API_BASE='http://127.0.0.1:11434/v1'          # Ollama OpenAI-compatible
cfg_path=pathlib.Path('data/config.yaml'); cfg=yaml.safe_load(cfg_path.read_text())
cfg['model']['name']=LLM_MODEL
cfg['model']['api_base']=API_BASE
cfg['rag']['stores_root']=STORES
cfg_path.write_text(yaml.safe_dump(cfg,sort_keys=False))
os.environ['OPENAI_API_KEY']='ollama'          # dummy; ignored
print('LLM   =',LLM_MODEL,'@',API_BASE)
print('embed =',cfg['rag']['embed_model'])

## 3 — Choose SCOPE → clone AOSP + HMI + VSS → index on GPU

`SCOPE`: `automotive` (start here) | `framework` (+frameworks/base) | `full` (bring your own).
Now clones **HMI** (Car UI Library `apps/Car/libs`) and **VSS** (COVESA vehicle_signal_specification,
dropped under `vendor/vss` so the VSS signal-tree chunker + customer priors pick it up).
Clone is non-fatal. Index runs on the free GPU (Ollama not up yet).

Note: the `automotive` scope already includes `vendor/`, so `vendor/vss` (COVESA) gets indexed.

## 3 — Choose SCOPE → clone AOSP → index on GPU

`SCOPE`: `automotive` (start here) | `framework` (+frameworks/base, big) | `full` (bring your own).
Clone is non-fatal (a repo that fails is skipped). Index runs on the free GPU (Ollama not up yet).

In [ ]:
import os, shutil, subprocess, pathlib
SCOPE='automotive'          # automotive | framework | full   (index scope, used in the indexer cell)
AOSP='/content/aosp'
shutil.rmtree(AOSP, ignore_errors=True)

G='https://android.googlesource.com/platform/'
REPOS={
  # core stack
  'hardware/interfaces':          G+'hardware/interfaces',
  'packages/services/Car':        G+'packages/services/Car',
  # FULL HMI — mọi app Car chính
  'packages/apps/Car/libs':       G+'packages/apps/Car/libs',        # Car UI Library
  'packages/apps/Car/Settings':   G+'packages/apps/Car/Settings',
  'packages/apps/Car/SystemUI':   G+'packages/apps/Car/SystemUI',
  'packages/apps/Car/Launcher':   G+'packages/apps/Car/Launcher',
  'packages/apps/Car/Cluster':    G+'packages/apps/Car/Cluster',
  'packages/apps/Car/Dialer':     G+'packages/apps/Car/Dialer',
  'packages/apps/Car/Media':      G+'packages/apps/Car/Media',
  'packages/apps/Car/Notification':G+'packages/apps/Car/Notification',
  'packages/apps/Car/Radio':      G+'packages/apps/Car/Radio',
  'packages/apps/Car/Messenger':  G+'packages/apps/Car/Messenger',
}
def clone(rel,url):
    dest=f'{AOSP}/{rel}'
    os.makedirs(pathlib.Path(dest).parent,exist_ok=True)
    r=subprocess.run(['git','clone','--depth','1',url,dest],capture_output=True,text=True)
    print('  cloned' if r.returncode==0 else '  SKIP', rel)
for rel,url in REPOS.items(): clone(rel,url)

# VSS (COVESA)
vss=f'{AOSP}/vendor/vss'
os.makedirs(pathlib.Path(vss).parent,exist_ok=True)
r=subprocess.run(['git','clone','--depth','1',
                  'https://github.com/COVESA/vehicle_signal_specification',vss])
print('  cloned VSS' if r.returncode==0 else '  SKIP VSS')

os.environ['AOSP_ROOT']=AOSP
print('SCOPE =',SCOPE)
print('--- tree ---');
!du -sh {AOSP}/packages/apps/Car/* {AOSP}/vendor/* 2>/dev/null

In [ ]:
%cd /content/android-auto-ai-agent
!git pull

In [ ]:
import os
if os.path.exists(f'{STORES}/_base/aosp15/manifest.json'):
    print('index already on Drive -> skip (delete manifest to rebuild / change scope)')
else:
    !python -m retrieval.indexer --aosp-root {AOSP} --base --aosp-version aosp15 --scope {SCOPE}
print('index at', f'{STORES}/_base/aosp15')

## 4 — Start Ollama + pull model
Ollama = prebuilt, **no CUDA/torch compile**. `start_ollama()` is idempotent — re-run anytime.

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os, time, subprocess, requests
from pathlib import Path

OLLAMA_LOG = "/content/ollama.log"
os.environ["OLLAMA_NUM_PARALLEL"] = "4"

def start_ollama():
    if not Path("/usr/local/bin/ollama").exists():
        print("Installing Ollama...")
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                       shell=True, capture_output=True)
    subprocess.run(["pkill", "-f", "ollama serve"], capture_output=True)
    time.sleep(1)
    subprocess.Popen(["ollama", "serve"],
                     stdout=open(OLLAMA_LOG, "w"), stderr=subprocess.STDOUT,
                     env={**os.environ})
    for i in range(30):
        try:
            r = requests.get("http://localhost:11434/api/tags", timeout=2)
            if r.status_code == 200:
                print(f"Ollama ready (took {i+1}s)")
                return True
        except Exception:
            pass
        time.sleep(1)
    raise RuntimeError("Ollama failed to start — check /content/ollama.log")

start_ollama()

In [ ]:
!/usr/local/bin/ollama pull qwen2.5-coder:32b
!/usr/local/bin/ollama list

## 5 — Run agent
Embedder on CPU (Ollama holds the GPU); query embedding is a few cheap calls.
Includes the per-layer specialist pass.

`--customer base` = base-only knowledge through the multi-tenant store: it loads
`stores/_base/aosp15` (built above). Without it the agent falls back to the empty
legacy index and retrieves nothing.

In [ ]:
BUG="Android 15: VSS Vehicle.Speed not updating in HMI after ignition ON"
!CUDA_VISIBLE_DEVICES="" python -m agent.main --bug "{BUG}" \
    --aosp-root /content/aosp --aosp-version aosp15 \
    --customer base --project default

---
### Notes
- **No compile anywhere** — Ollama ships prebuilt binaries; avoids the vLLM/torch/flashinfer
  version+JIT-compile issues on Colab Python 3.13.
- **New session** → cells 0,1,2,3 (index skips, reused from Drive), 4 (ollama), 5.
- **Ollama died** → re-run cell 4 (`start_ollama()`); restarts only if needed.
- **Change coverage** → set SCOPE in cell 3, delete `stores/_base/aosp15/manifest.json`, re-run cell 3.
- **Lighter model if 32B is slow/tight** → set LLM_MODEL='qwen2.5-coder:14b' in cell 2 and
  `ollama pull qwen2.5-coder:14b` in cell 4.